# W&B Sweeps — Fine-tune the Real HiLift KPI Model

Sweeps are W&B's hyperparameter search. This notebook fine-tunes the `TinyTransolverKpi` checkpoint produced by `experiment_tracking.ipynb`.

The loop is intentionally compact:

1. Pull `datasets:production` and `checkpoints:production` from the Registry.
2. Start every trial from the same `TinyTransolverKpi` champion weights.
3. Sweep optimizer hyperparameters for a short fine-tune.
4. Log every trial checkpoint as a model artifact candidate.
5. Promote the best trial into `model_hyperparameter_tuned:production`.

**Prerequisites**
- `registry.ipynb` produces `wandb-registry-high_lift_surrogates/datasets:production`.
- `experiment_tracking.ipynb` produces `wandb-registry-high_lift_surrogates/checkpoints:production`.


## 1. Setup

Third-party imports and the section-local constants — no
`physicsx_demo` imports, notebook stands alone.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "physicsx_demo").exists())
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import importlib
import math
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import wandb

from physicsx_demo.utils.env import load_local_env, resolve_wandb_target
from physicsx_demo.utils.hilift_kpi import HILIFT_TARGET_COLUMNS, HiLiftKpiDataset, denormalize_kpi_targets, prepare_hilift_kpi_bundle


import physicsx_demo.utils.model as model_utils
model_utils = importlib.reload(model_utils)
TinyTransolverKpi = model_utils.TinyTransolverKpi
count_params = model_utils.count_params

load_local_env()

device = torch.device("cpu")
ENTITY, PROJECT = resolve_wandb_target()

REGISTRY_NAME = "high_lift_surrogates"
DATASET_COLLECTION = "datasets"
BASE_COLLECTION = "checkpoints"
TUNED_COLLECTION = "model_hyperparameter_tuned"

DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
CHECKPOINT_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{BASE_COLLECTION}:production"

print(f"torch  = {torch.__version__}")
print(f"wandb  = {wandb.__version__}")
print(f"target = {ENTITY}/{PROJECT}")
print(f"device = {device}")


## 2. Model — TinyTransolverKpi

The sweep fine-tunes the same KPI architecture produced by the training notebook. The checkpoint contains the config, state dict, normalization stats, and target column names.


In [ ]:
# TinyTransolverKpi and count_params are imported from `physicsx_demo.utils.model`.


## 3. Pull the Pretrained Weights + Dataset from W&B

A short `sweep-setup` run consumes both Registry entries so W&B lineage records the warm-start and data dependencies before the agent starts trials.


In [ ]:
def rmse_by_target(pred, true, prefix="val"):
    return {
        f"{prefix}/{name}_rmse": float(np.sqrt(np.mean((pred[:, idx] - true[:, idx]) ** 2)))
        for idx, name in enumerate(HILIFT_TARGET_COLUMNS)
    }


def evaluate(model, loader, stats):
    model.eval()
    losses = []
    preds = []
    targets = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            losses.append(F.mse_loss(pred, yb).item())
            preds.append(pred.cpu())
            targets.append(yb.cpu())
    pred_norm = torch.cat(preds, dim=0)
    target_norm = torch.cat(targets, dim=0)
    pred_phys = denormalize_kpi_targets(pred_norm, stats)
    target_phys = denormalize_kpi_targets(target_norm, stats)
    return float(np.mean(losses)), rmse_by_target(pred_phys, target_phys), pred_phys, target_phys


with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="sweep-setup",
    name="kpi-sweep-setup",
    tags=["notebook", "sweep", "setup", "real-hiliftaeroml"],
) as setup_run:
    ckpt_art = setup_run.use_artifact(CHECKPOINT_REGISTRY_PATH)
    ckpt_dir = Path(ckpt_art.download())
    ds_art = setup_run.use_artifact(DATASET_REGISTRY_PATH)
    ds_dir = Path(ds_art.download())

    ckpt_file = next(ckpt_dir.glob("*.pt"))
    ckpt = torch.load(ckpt_file, map_location="cpu")

    MODEL_KEYS = ("in_dim", "out_dim", "hidden", "n_layers", "n_slices", "n_heads", "ffn_expansion", "dropout", "use_global_context")
    PRETRAINED_CONFIG = {k: ckpt["config"][k] for k in MODEL_KEYS if k in ckpt["config"]}
    PRETRAINED_STATE_DICT = ckpt["state_dict"]
    SEED = int(ckpt["config"].get("seed", 42))
    N_GEOMETRIES = int(ckpt["config"].get("n_geometries", 80))

    bundle = prepare_hilift_kpi_bundle(ds_dir, n_geometries=N_GEOMETRIES, seed=SEED)
    STATS = bundle.stats
    TRAIN_DATASET = HiLiftKpiDataset(bundle.train, STATS)
    VAL_LOADER = DataLoader(HiLiftKpiDataset(bundle.val, STATS), batch_size=128)

    print(f"loaded champion checkpoint: epoch={ckpt.get('epoch')} val_mse={ckpt.get('val_mse'):.5f}")
    print(f"model config: {PRETRAINED_CONFIG}")
    print(f"train rows={len(bundle.train)} val rows={len(bundle.val)}")

    baseline_model = TinyTransolverKpi(**PRETRAINED_CONFIG).to(device)
    baseline_model.load_state_dict(PRETRAINED_STATE_DICT)
    baseline_mse, baseline_rmse, _, _ = evaluate(baseline_model, VAL_LOADER, STATS)
    BASELINE_SELECTION_SCORE = baseline_mse
    print(f"warm-start val_mse={baseline_mse:.5f} {baseline_rmse}")


## 4. Sweep Configuration and Training Function

The architecture is fixed by the loaded checkpoint. Trials fine-tune optimizer settings only: learning rate, weight decay, and batch size.


In [ ]:
FINE_TUNE_EPOCHS = 5

SWEEP_CONFIG = {
    "method": "bayes",
    "metric": {"name": "val/selection_score", "goal": "minimize"},
    "parameters": {
        "lr": {"distribution": "log_uniform_values", "min": 1e-5, "max": 3e-3},
        "weight_decay": {"distribution": "log_uniform_values", "min": 1e-6, "max": 1e-3},
        "batch_size": {"values": [32, 64, 128]},
    },
}


def train_fn():
    with wandb.init(job_type="sweep-trial", tags=["notebook", "sweep", "fine-tune", "real-hiliftaeroml"]) as run:
        cfg = run.config
        run.use_artifact(CHECKPOINT_REGISTRY_PATH)
        run.use_artifact(DATASET_REGISTRY_PATH)

        torch.manual_seed(SEED)
        model = TinyTransolverKpi(**PRETRAINED_CONFIG).to(device)
        model.load_state_dict(PRETRAINED_STATE_DICT)
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        train_loader = DataLoader(TRAIN_DATASET, batch_size=cfg.batch_size, shuffle=True)

        val_mse = float("nan")
        val_rmse = {}
        for epoch in range(1, FINE_TUNE_EPOCHS + 1):
            model.train()
            losses = []
            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                pred = model(xb)
                loss = F.mse_loss(pred, yb)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                losses.append(loss.item())

            val_mse, val_rmse, _, _ = evaluate(model, VAL_LOADER, STATS)
            run.log({
                "epoch": epoch,
                "train/loss": float(np.mean(losses)),
                "val/mse": val_mse,
                "val/selection_score": val_mse,
                "model/improvement_vs_warm_start": BASELINE_SELECTION_SCORE - val_mse,
                **val_rmse,
            })

        ckpt_path = Path(tempfile.mkdtemp()) / "tiny_transolver_kpi_tuned.pt"
        torch.save(
            {
                "state_dict": model.state_dict(),
                "config": {**PRETRAINED_CONFIG, "seed": SEED, "n_geometries": N_GEOMETRIES},
                "stats": STATS.to_dict(),
                "target_columns": HILIFT_TARGET_COLUMNS,
                "val_mse": val_mse,
                "val_rmse": val_rmse,
                "base_checkpoint": CHECKPOINT_REGISTRY_PATH,
            },
            ckpt_path,
        )
        artifact = wandb.Artifact(
            "high_lift_kpi_surrogate_ckpt_tuned",
            type="model",
            description="Fine-tuned TinyTransolverKpi candidate from W&B sweep.",
            metadata={"val_mse": val_mse, "selection_score": val_mse, "n_params": count_params(model), **val_rmse},
        )
        artifact.add_file(str(ckpt_path))
        logged = run.log_artifact(artifact, aliases=["candidate", f"run-{run.id}"])
        run.summary["model_checkpoint/artifact"] = f"{ENTITY}/{PROJECT}/{logged.name}:candidate"
        run.summary["val/selection_score"] = val_mse
        run.summary["val/mse"] = val_mse
        for key, value in val_rmse.items():
            run.summary[key] = value


## 5. Launch the Sweep

Each trial logs a `high_lift_kpi_surrogate_ckpt_tuned` artifact candidate. The promotion cell links the best candidate into the tuned Registry Collection.


In [ ]:
sweep_id = wandb.sweep(SWEEP_CONFIG, entity=ENTITY, project=PROJECT)

print(f"sweep id  = {sweep_id}")
print(f"sweep url = https://wandb.ai/{ENTITY}/{PROJECT}/sweeps/{sweep_id}")

wandb.agent(sweep_id, function=train_fn, count=8)


## 6. Promote the Winner into a New Registry Collection

The best trial is the one with the lowest final `val/selection_score`, which is the validation MSE on normalized `cl/cd/cm` targets.


In [ ]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="sweep-promotion",
    name="sweep-winner-promotion",
    tags=["notebook", "sweep", "promotion", "real-hiliftaeroml"],
) as promo_run:
    api = wandb.Api(overrides={"entity": ENTITY, "project": PROJECT})
    sweep = api.sweep(f"{ENTITY}/{PROJECT}/{sweep_id}")

    def trial_selection_score(run):
        value = run.summary.get("val/selection_score", float("inf"))
        return float(value) if value is not None else float("inf")

    viable_runs = [r for r in sweep.runs if math.isfinite(trial_selection_score(r))]
    if not viable_runs:
        raise RuntimeError("No completed sweep runs with finite val/selection_score were found.")

    best = min(viable_runs, key=trial_selection_score)
    best_score = trial_selection_score(best)
    print(f"winner: {best.name}   val/selection_score = {best_score:.5f}   warm-start score was {BASELINE_SELECTION_SCORE:.5f}")

    tuned_artifacts = [
        artifact for artifact in best.logged_artifacts()
        if artifact.type == "model" and artifact.name.startswith("high_lift_kpi_surrogate_ckpt_tuned")
    ]
    if not tuned_artifacts:
        raise RuntimeError(f"Best run {best.name} did not log high_lift_kpi_surrogate_ckpt_tuned.")
    best_art = tuned_artifacts[-1]
    best_art.wait()

    top_rows = []
    for rank, trial in enumerate(sorted(viable_runs, key=trial_selection_score), start=1):
        top_rows.append([
            rank,
            trial.name,
            trial.id,
            trial_selection_score(trial),
            trial.summary.get("val/cl_rmse"),
            trial.summary.get("val/cd_rmse"),
            trial.summary.get("val/cm_rmse"),
            trial.config.get("lr"),
            trial.config.get("weight_decay"),
            trial.config.get("batch_size"),
        ])
    promo_run.log({
        "sweep/top_trials": wandb.Table(
            columns=["rank", "run_name", "run_id", "val_selection_score", "cl_rmse", "cd_rmse", "cm_rmse", "lr", "weight_decay", "batch_size"],
            data=top_rows,
        )
    })

    target_path = f"wandb-registry-{REGISTRY_NAME}/{TUNED_COLLECTION}"
    aliases = ["production", "sweep-winner", f"sweep-{sweep_id}", f"run-{best.id}"]
    promo_run.link_artifact(best_art, target_path=target_path, aliases=aliases)
    promo_run.summary["sweep/winner_run"] = best.name
    promo_run.summary["sweep/winner_run_id"] = best.id
    promo_run.summary["sweep/winner_selection_score"] = best_score
    promo_run.summary["sweep/warm_start_selection_score"] = BASELINE_SELECTION_SCORE
    promo_run.summary["model_registry/path"] = target_path
    promo_run.summary["model_registry/aliases"] = aliases
    print(f"linked -> {target_path} aliases={aliases}")


## 7. Wrap Up — What to Look at in the W&B UI

**Sweep page**
- Parallel-coordinates plot across `lr`, `weight_decay`, and `batch_size`.
- Trial runs with `val/mse`, `val/cl_rmse`, `val/cd_rmse`, and `val/cm_rmse`.

**Lineage graph**
- Every trial consumes both Registry entries: `checkpoints:production` and `datasets:production`.

**Registry**
- `datasets` — real HiLiftAeroML KPI CSV artifact.
- `checkpoints` — base `TinyTransolverKpi` from `experiment_tracking.ipynb`.
- `model_hyperparameter_tuned` — sweep winner checkpoint aliased `production`, `sweep-winner`, `sweep-<id>`, and `run-<id>`.
